# 🔧 PromptCanary — Writing Custom Probes

This notebook covers everything you need to know to write production-quality custom probes.

**Topics:**
1. The `@probe` decorator (functional API)
2. `BaseProbe` subclass (class API — recommended for complex probes)
3. Partial scoring patterns
4. Metadata best practices
5. Domain-specific probe examples: tool calls, JSON schema, semantic checks
6. Using custom probes in YAML config
7. Testing your probes

In [ ]:
!pip install promptcanary

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').parent))

from promptcanary.core.models import (
    CanaryPrompt, LLMResponse, ProbeCategory, ProbeResult
)
from promptcanary.core.probes.base import BaseProbe, probe, get_probe_registry
from promptcanary import CanarySuite, FileBaselineStore
from promptcanary.core.models import ProviderConfig
from promptcanary.providers.base import BaseLLMProvider

print('✅ Ready')

✅ Ready


## 1. Functional API — `@probe` Decorator

Best for simple, single-purpose probes.

In [3]:
@probe("contains_code_fence", name="Contains Code Fence", category=ProbeCategory.FORMAT)
def check_code_fence(prompt: CanaryPrompt, response: LLMResponse) -> ProbeResult:
    """Check that the response contains a markdown code fence."""
    has_fence = '```' in response.content
    return ProbeResult(
        probe_id="contains_code_fence",
        probe_name="Contains Code Fence",
        category=ProbeCategory.FORMAT,
        prompt_id=prompt.id,
        passed=has_fence,
        score=1.0 if has_fence else 0.0,
        details="Code fence found." if has_fence else "No code fence in response.",
    )

# Test it directly:
p = CanaryPrompt(id="p1", text="Show Python code to print Hello.")
r_with = LLMResponse(prompt_id="p1", provider_model_id="test", content="```python\nprint('Hello')\n```")
r_without = LLMResponse(prompt_id="p1", provider_model_id="test", content="Use print('Hello')")

instance = check_code_fence()
print('With fence   :', instance(p, r_with).passed, instance(p, r_with).score)
print('Without fence:', instance(p, r_without).passed, instance(p, r_without).score)

With fence   : True 1.0
Without fence: False 0.0


## 2. Class API — `BaseProbe` Subclass

Recommended for probes with configuration, state, or complex logic.

In [4]:
import json
import re
from typing import Any

class ToolCallProbe(BaseProbe):
    """Verifies that a response contains a specific tool/function call.

    Designed for agent workflows where the model should call a specific
    function with valid arguments.

    Args:
        function_name:   Name of the function that must be called.
        required_args:   Argument keys that must be present in the call.

    Score:
        1.0  — function found with all required args
        0.5  — function found but args incomplete
        0.0  — function not found
    """

    probe_id = "tool_call"
    name = "Tool Call"
    category = ProbeCategory.CUSTOM
    description = "Verifies the model calls the expected function with required args."

    def __init__(
        self,
        function_name: str,
        required_args: list[str] | None = None,
    ) -> None:
        self.function_name = function_name
        self.required_args = required_args or []

    def evaluate(self, prompt: CanaryPrompt, response: LLMResponse) -> ProbeResult:
        content = response.content

        # Try to find the function call in the content
        # Supports both JSON tool_call format and plain text mentions
        function_found = self.function_name in content

        if not function_found:
            return self._make_result(
                prompt.id, passed=False, score=0.0,
                details=f"Function '{self.function_name}' not found in response.",
                metadata={"function_name": self.function_name, "found": False},
            )

        # Check for required args
        if not self.required_args:
            return self._make_result(
                prompt.id, passed=True, score=1.0,
                details=f"Function '{self.function_name}' found.",
                metadata={"function_name": self.function_name, "found": True},
            )

        missing = [arg for arg in self.required_args if arg not in content]
        score = (len(self.required_args) - len(missing)) / len(self.required_args)
        passed = not missing

        return self._make_result(
            prompt.id,
            passed=passed,
            score=score if function_found else 0.0,
            details=(
                f"Function '{self.function_name}' found. "
                + (f"Missing args: {missing}" if missing else "All required args present.")
            ),
            metadata={
                "function_name": self.function_name,
                "required_args": self.required_args,
                "missing_args": missing,
            },
        )


# Test it:
probe_instance = ToolCallProbe("search", required_args=["query", "limit"])

good_response = LLMResponse(
    prompt_id="p1", provider_model_id="test",
    content='{"function": "search", "args": {"query": "Paris weather", "limit": 5}}'
)
partial_response = LLMResponse(
    prompt_id="p1", provider_model_id="test",
    content='{"function": "search", "args": {"query": "Paris weather"}}'
)
bad_response = LLMResponse(
    prompt_id="p1", provider_model_id="test",
    content="I'll look that up for you."
)

p = CanaryPrompt(id="p1", text="Search for Paris weather.")

for label, resp in [("Full", good_response), ("Partial", partial_response), ("Missing", bad_response)]:
    r = probe_instance(p, resp)
    print(f'{label:10s}: passed={str(r.passed):5s}  score={r.score:.2f}  details={r.details}')

Full      : passed=True   score=1.00  details=Function 'search' found. All required args present.
Partial   : passed=False  score=0.50  details=Function 'search' found. Missing args: ['limit']
Missing   : passed=False  score=0.00  details=Function 'search' not found in response.


## 3. Partial Scoring Patterns

Partial scores (between 0.0 and 1.0) are critical for trend tracking — they let you detect *gradual* drift before it becomes a hard failure.

In [5]:
class SentenceCountProbe(BaseProbe):
    """Checks that the response has approximately the expected number of sentences.

    Uses partial scoring to detect gradual verbosity drift.

    Score: 1.0 when within tolerance, degrades linearly outside it.
    """
    probe_id = "sentence_count"
    name = "Sentence Count"
    category = ProbeCategory.REASONING

    def __init__(self, expected: int, tolerance: float = 0.5) -> None:
        self.expected = expected
        self.tolerance = tolerance

    def evaluate(self, prompt: CanaryPrompt, response: LLMResponse) -> ProbeResult:
        # Simple sentence count heuristic
        sentences = [s.strip() for s in re.split(r'[.!?]+', response.content) if s.strip()]
        count = len(sentences)

        ratio = count / max(self.expected, 1)
        deviation = abs(ratio - 1.0)

        # Linear decay outside tolerance band
        if deviation <= self.tolerance:
            score = 1.0 - (deviation / self.tolerance) * 0.2  # at most 20% penalty within tolerance
        else:
            score = max(0.0, 1.0 - deviation)

        passed = deviation <= self.tolerance

        return self._make_result(
            prompt.id,
            passed=passed,
            score=score,
            details=(
                f"{count} sentence(s) vs expected ~{self.expected} "
                f"({ratio:.1%} of expected, tolerance ±{self.tolerance:.0%})."
            ),
            metadata={"sentence_count": count, "expected": self.expected, "ratio": ratio},
        )

probe_s = SentenceCountProbe(expected=2, tolerance=0.5)
p = CanaryPrompt(id="p1", text="Test")

for text, label in [
    ("This is one sentence.", "1 sentence"),
    ("First sentence. Second sentence.", "2 sentences (expected)"),
    ("S1. S2. S3.", "3 sentences"),
    ("S1. S2. S3. S4. S5. S6. S7. S8.", "8 sentences (big drift)"),
]:
    r_resp = LLMResponse(prompt_id="p1", provider_model_id="test", content=text)
    r = probe_s(p, r_resp)
    print(f'{label:35s}: score={r.score:.3f}  passed={r.passed}')

1 sentence                         : score=0.800  passed=True
2 sentences (expected)             : score=1.000  passed=True
3 sentences                        : score=0.800  passed=True
8 sentences (big drift)            : score=0.000  passed=False


## 4. Using Custom Probes in a Suite

In [6]:
class MockProviderForCustom(BaseLLMProvider):
    def __init__(self):
        super().__init__(ProviderConfig(model_id="mock/v1"))
    def complete(self, prompt, *, system_prompt=None):
        return LLMResponse(
            prompt_id=prompt.id, provider_model_id="mock/v1",
            content='{"function": "search", "args": {"query": "test", "limit": 5}}',
            finish_reason="stop", latency_ms=10.0,
        )

suite = CanarySuite(
    name="custom-probe-suite",
    prompts=[
        CanaryPrompt(id="p1", text="Search for the latest news about AI."),
    ],
    probes=[
        ToolCallProbe("search", required_args=["query", "limit"]),
        SentenceCountProbe(expected=1, tolerance=1.0),
    ],
)

from promptcanary.core.reporter import Reporter
result = suite.run(MockProviderForCustom(), show_progress=False)
Reporter(result).print_terminal()

╭──────────────────────────────────────────── PromptCanary Run Report ────────────────────────────────────────────╮
│ custom-probe-suite  ·  Score: 100.0%  ·  Pass rate: 100.0%  ·  Provider: mock/v1  ·  Probes: 2                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────┬───────────┬────────┬────────┬───────┬────────────────────────────────────────────────╮
│ Probe                    │ Category  │ Prompt │ Result │ Score │ Details                                        │
├──────────────────────────┼───────────┼────────┼────────┼───────┼────────────────────────────────────────────────┤
│ Tool Call                │ custom    │ p1     │ ✓ PASS │  1.00 │ Function 'search' found. All required args     │
│                          │           │        │        │       │ present.                                       │
│ Sentence Count           │ reasoning │ p1     │ ✓ PASS │  1.00 │ 1 sentence(s) vs expected ~1 (100.0% of        │
│                          │           │        │        │       │ expected, tolerance ±100%).                    │
╰──────────────────────────┴───────────┴────────┴────────┴───────┴────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ All probes passed.                                                                                           │
│ Overall score: 100.0%  ·  Duration: 0ms  ·  Run ID: 80fa6629-7101-4cec-9e70-b74440ae4caf                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 5. Checking the Registry

Every `BaseProbe` subclass with a non-empty `probe_id` is auto-registered.

In [7]:
registry = get_probe_registry()
print(f'Total registered probes: {len(registry)}\n')
for probe_id, cls in sorted(registry.items()):
    print(f'  {probe_id:30s}  →  {cls.__name__}')

Total registered probes: 22

  confidence_language             →  ConfidenceLanguageProbe
  contains_code_fence             →  Contains Code Fence
  direct_answer                   →  DirectAnswerProbe
  expected_keywords               →  ExpectedKeywordsProbe
  factual_consistency             →  FactualConsistencyProbe
  json_key_order                  →  JsonKeyOrderProbe
  json_schema                     →  JsonSchemaProbe
  json_validity                   →  JsonValidityProbe
  keyword_presence                →  KeywordPresenceProbe
  markdown_headers                →  MarkdownHeaderProbe
  refusal                         →  RefusalProbe
  response_length                 →  ResponseLengthProbe
  safety_language                 →  SafetyLanguageProbe
  sentence_count                  →  SentenceCountProbe
  sentiment                       →  SentimentProbe
  step_by_step                    →  StepByStepProbe
  tool_call                       →  ToolCallProbe
  tool_call_args        

## 6. Testing Your Probes

A well-written probe has tests for: happy path, failure path, partial score, and edge cases.

In [8]:
def run_probe_tests(probe_instance: BaseProbe, test_cases: list[dict]) -> None:
    """Lightweight test runner for probe validation in notebooks."""
    print(f'Testing: {probe_instance.name} ({probe_instance.probe_id})')
    print('-' * 60)
    all_passed = True
    for tc in test_cases:
        prompt = CanaryPrompt(id="tp", text=tc.get('prompt_text', 'test'))
        response = LLMResponse(prompt_id="tp", provider_model_id="test", content=tc['response'])
        result = probe_instance(prompt, response)

        ok = True
        if 'expect_passed' in tc:
            ok = ok and (result.passed == tc['expect_passed'])
        if 'expect_score_gte' in tc:
            ok = ok and (result.score >= tc['expect_score_gte'])
        if 'expect_score_lte' in tc:
            ok = ok and (result.score <= tc['expect_score_lte'])

        status = '✅' if ok else '❌'
        if not ok:
            all_passed = False
        print(f'  {status} {tc["name"]:30s}  passed={result.passed}  score={result.score:.2f}')

    print()
    print('All tests passed!' if all_passed else '⚠️  Some tests failed!')


# Test the SentenceCountProbe:
run_probe_tests(
    SentenceCountProbe(expected=2, tolerance=0.5),
    [
        {"name": "Exactly 2 sentences",    "response": "Sentence one. Sentence two.",  "expect_passed": True,  "expect_score_gte": 0.8},
        {"name": "3 sentences (in range)", "response": "A. B. C.",                     "expect_passed": True},
        {"name": "8 sentences (way off)",  "response": "A. B. C. D. E. F. G. H.",     "expect_passed": False, "expect_score_lte": 0.5},
        {"name": "Empty response",         "response": "",                             "expect_passed": False},
    ]
)

Testing: Sentence Count (sentence_count)
------------------------------------------------------------
  ✅ Exactly 2 sentences             passed=True  score=1.00
  ✅ 3 sentences (in range)          passed=True  score=0.80
  ✅ 8 sentences (way off)           passed=False  score=0.00
  ✅ Empty response                  passed=False  score=0.00

All tests passed!


---

## Probe Writing Checklist

Before submitting a probe to the PromptCanary repo:

- [ ] Unique `probe_id` in snake_case
- [ ] Meaningful `name` and `category`
- [ ] Docstring with Args, Score semantics, and Example
- [ ] Returns partial scores (not just 0/1) where meaningful
- [ ] Wraps risky code in try/except (never raises from `evaluate()`)
- [ ] Populates `metadata` with probe-specific diagnostic data
- [ ] Tests for happy path, failure path, and at least one edge case
- [ ] Added to `CHANGELOG.md` under `[Unreleased]`

See [CONTRIBUTING.md](../CONTRIBUTING.md) for the full guide.